In [1]:
"""
# ========================================
# 掌骨X光小关节识别 - 领域自适应 + YOLOv8X
# ========================================

## 1. 依赖安装和导入
"""
!pip install ultralytics

import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split, KFold
from ultralytics import YOLO
import matplotlib.pyplot as plt
import time
import glob
from tqdm import tqdm
import random
import yaml
from pathlib import Path
import math
import shutil
import pandas as pd
import json

print("✅ 依赖库导入完成")

# 设置随机种子
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

# ==================== Transformer位置编码模块（步骤8）====================
class PositionalEncoding2D(nn.Module):
    """
    2D位置编码，使用 a*x + y 形式表示像素点位置信息
    根据transformer论文和思考内容实现
    """
    def __init__(self, d_model, height, width, a=1.0):
        """
        初始化2D位置编码
        
        Args:
            d_model: 嵌入维度
            height: 图像高度
            width: 图像宽度
            a: 缩放因子，用于平衡x和y的权重
        """
        super(PositionalEncoding2D, self).__init__()
        self.d_model = d_model
        self.height = height
        self.width = width
        self.a = a
        
        # 创建位置编码矩阵
        self.pe = self._create_positional_encoding(height, width, d_model, a)
    
    def _create_positional_encoding(self, height, width, d_model, a):
        """
        创建2D位置编码矩阵
        
        使用 a*x + y 形式表示位置，然后映射到d_model维度
        """
        pe = np.zeros((height, width, d_model))
        
        # 创建坐标网格
        y_coords, x_coords = np.meshgrid(range(height), range(width), indexing='ij')
        
        # 使用 a*x + y 形式的位置编码
        position = a * x_coords + y_coords
        
        # 映射到d_model维度（使用sin/cos编码）
        for i in range(d_model):
            if i % 2 == 0:
                pe[:, :, i] = np.sin(position / (10000 ** (i / d_model)))
            else:
                pe[:, :, i] = np.cos(position / (10000 ** ((i - 1) / d_model)))
        
        return torch.from_numpy(pe).float()
    
    def forward(self, x):
        """
        添加位置编码到输入特征
        
        Args:
            x: 输入张量，形状为 (batch, channels, height, width)
        
        Returns:
            添加位置编码后的张量
        """
        batch_size, channels, height, width = x.size()
        
        # 如果输入尺寸与初始化尺寸不同，重新创建位置编码
        if height != self.height or width != self.width:
            self.pe = self._create_positional_encoding(height, width, self.d_model, self.a).to(x.device)
            self.height = height
            self.width = width
        
        # 将位置编码添加到输入特征
        pe = self.pe.unsqueeze(0).expand(batch_size, -1, -1, -1).to(x.device)
        
        # 如果通道数不匹配，使用线性投影
        if channels != self.d_model:
            projection = nn.Linear(self.d_model, channels).to(x.device)
            pe = projection(pe.permute(0, 3, 1, 2)).permute(0, 2, 3, 1)
        
        return x + pe.permute(0, 3, 1, 2)


class TransformerEncoder(nn.Module):
    """
    Transformer编码器模块，用于处理带有位置编码的图像特征
    """
    def __init__(self, in_channels, num_layers=2, num_heads=4, d_model=256):
        """
        初始化Transformer编码器
        
        Args:
            in_channels: 输入通道数
            num_layers: Transformer层数
            num_heads: 注意力头数
            d_model: 模型维度
        """
        super(TransformerEncoder, self).__init__()
        
        self.d_model = d_model
        self.in_channels = in_channels
        
        # 输入投影
        self.input_projection = nn.Conv2d(in_channels, d_model, kernel_size=1)
        
        # 位置编码
        self.pos_encoding = PositionalEncoding2D(d_model, height=640, width=640, a=1.0)
        
        # Transformer编码器层
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=d_model * 4,
            dropout=0.1,
            activation='gelu',
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 输出投影
        self.output_projection = nn.Conv2d(d_model, in_channels, kernel_size=1)
    
    def forward(self, x):
        """
        前向传播
        
        Args:
            x: 输入特征，形状为 (batch, channels, height, width)
        
        Returns:
            编码后的特征，形状为 (batch, channels, height, width)
        """
        batch_size, channels, height, width = x.size()
        
        # 输入投影
        x = self.input_projection(x)
        
        # 添加位置编码
        x = self.pos_encoding(x)
        
        # 转换为序列格式 (batch, seq_len, d_model)
        x = x.flatten(2).transpose(1, 2)
        
        # 通过Transformer编码器
        x = self.transformer_encoder(x)
        
        # 转换回空间格式 (batch, d_model, height, width)
        x = x.transpose(1, 2).view(batch_size, self.d_model, height, width)
        
        # 输出投影
        x = self.output_projection(x)
        
        return x


# ==================== GFR（高斯场域适应）模块====================
class GFRDomainAdaptation(nn.Module):
    """
    GFR（高斯场域适应）层
    域判别器：只需要判断是否是一个域
    源域标签: 0，目标域标签: 1
    """
    def __init__(self, input_dim, hidden_dim=512, kernel_type='rbf', kernel_params=None):
        super(GFRDomainAdaptation, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.kernel_type = kernel_type
        self.kernel_params = kernel_params if kernel_params else {'sigma': 1.0}
        
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.5)
        )
        # 域分类器：输出2个类别（源域/目标域）
        self.domain_classifier = nn.Linear(hidden_dim, 2)
    
    def gaussian_kernel(self, x1, x2, sigma=1.0):
        """高斯核函数"""
        x1_sq = torch.sum(x1 ** 2, dim=1, keepdim=True)
        x2_sq = torch.sum(x2 ** 2, dim=1, keepdim=True)
        dist_sq = x1_sq + x2_sq.t() - 2 * torch.mm(x1, x2.t())
        return torch.exp(-dist_sq / (2 * sigma ** 2))
    
    def forward(self, source_features, target_features):
        """
        前向传播
        
        Args:
            source_features: 源域特征
            target_features: 目标域特征
        
        Returns:
            domain_loss: 域分类损失
            domain_acc: 域分类准确率
        """
        batch_size = source_features.size(0)
        
        # 提取特征
        source_hidden = self.feature_extractor(source_features)
        target_hidden = self.feature_extractor(target_features)
        
        # 域分类
        source_domain_pred = self.domain_classifier(source_hidden)
        target_domain_pred = self.domain_classifier(target_hidden)
        
        # 源域标签为0，目标域标签为1
        source_domain_labels = torch.zeros(batch_size, dtype=torch.long).to(source_features.device)
        target_domain_labels = torch.ones(batch_size, dtype=torch.long).to(target_features.device)
        
        # 计算域分类损失
        domain_loss = F.cross_entropy(source_domain_pred, source_domain_labels) + \
                      F.cross_entropy(target_domain_pred, target_domain_labels)
        
        # 计算域分类准确率
        source_pred_cls = torch.argmax(source_domain_pred, dim=1)
        target_pred_cls = torch.argmax(target_domain_pred, dim=1)
        
        source_acc = (source_pred_cls == source_domain_labels).float().mean()
        target_acc = (target_pred_cls == target_domain_labels).float().mean()
        domain_acc = (source_acc + target_acc) / 2
        
        return domain_loss, domain_acc


"""
## 2. 配置和参数设置
"""
# ==================== 配置 ====================
class Config:
    def __init__(self):
        # ========== 数据集路径配置 ==========
        # 说明：classes是一级目录，二级目录是对应分级
        
        # classes目录结构
        self.classes_dir = '训练代码/classes'  # classes一级目录
        self.train_dir = os.path.join(self.classes_dir, 'train')  # 训练集分级
        self.val_dir = os.path.join(self.classes_dir, 'val')  # 验证集分级
        self.test_dir = os.path.join(self.classes_dir, 'test')  # 测试集分级
        
        # 目标域数据（图片文件夹，用于泛化测试）
        self.target_images = '训练代码/photos'  # 目标域图片文件夹路径
        self.source_images = '训练代码/photos/arthrosis'  # 源域arthrosis目录
        
        # 类别配置
        self.num_classes = 10  # 小关节类别数量
        self.class_names = ['joint_1', 'joint_2', 'joint_3', 'joint_4', 'joint_5', 
                           'joint_6', 'joint_7', 'joint_8', 'joint_9', 'joint_10']
        self.classes_csv = None  # 类别CSV文件（可选）
        
        # ========== 模型配置 ==========
        self.model_path = 'models/best.pt'
        self.output_dir = 'output'
        self.yolo_model = 'yolov8s.pt'
        self.imgsz = 640
        
        # ========== 训练配置 ==========
        self.batch_size = 16
        self.epochs = 100
        self.lr = 1e-4
        self.fold = 5
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        # ========== 领域自适应配置 ==========
        self.domain_adaptation = True  # 启用领域自适应
        self.lambda_da = 0.1  # 领域自适应损失权重
        self.domain_test_interval = 10  # 每隔多少epoch测试一次域分类准确率
        
        # ========== 其他配置 ==========
        self.use_gan = True
        self.lambda_gan = 0.01
        self.alpha = 0.1

config = Config()

# 创建输出目录
os.makedirs(config.output_dir, exist_ok=True)

# 掌骨数据集类（源域，有标注）
class MetacarpalDataset(Dataset):
    """
    掌骨数据集加载器
    
    支持两种模式：
    1. 直接传入image_paths列表
    2. 从config.source_images或config.target_images加载
    """
    def __init__(self, image_paths=None, transform=None, auto_load=False, use_source=True):
        """
        初始化掌骨数据集
        
        Args:
            image_paths: 图像路径列表（可选）
            transform: 数据增强
            auto_load: 是否自动从config路径加载
            use_source: True使用源域(source_images)，False使用目标域(target_images)
        """
        if auto_load:
            # 自动从文件夹加载
            self.image_paths = []
            self.label_paths = None
            
            # 根据use_source选择加载源域或目标域
            image_dir = config.source_images if use_source else config.target_images
            
            # 获取所有图片文件
            for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG']:
                self.image_paths.extend(glob.glob(os.path.join(image_dir, ext)))
            
            # 排序保证顺序一致
            
            # 排序保证顺序一致
            sorted_pairs = sorted(zip(self.image_paths, self.label_paths), 
                                  key=lambda x: os.path.basename(x[0]))
            self.image_paths, self.label_paths = zip(*sorted_pairs) if sorted_pairs else ([], [])
        else:
            self.image_paths = image_paths if image_paths else []
            self.label_paths = label_paths if label_paths else []
        
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        # 读取图像
        image = cv2.imread(self.image_paths[idx])
        if image is None:
            image = np.zeros((config.imgsz, config.imgsz, 3), dtype=np.uint8)
        
        # 读取标注
        label_path = self.label_paths[idx]
        boxes = []
        labels = []
        
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        class_id = int(parts[0])
                        x_center, y_center, width, height = map(float, parts[1:5])
                        
                        # 转换为绝对坐标
                        img_h, img_w = image.shape[:2]
                        x_center *= img_w
                        y_center *= img_h
                        width *= img_w
                        height *= img_h
                        
                        # 转换为边界框格式 [x1, y1, x2, y2]
                        x1 = x_center - width / 2
                        y1 = y_center - height / 2
                        x2 = x_center + width / 2
                        y2 = y_center + height / 2
                        
                        boxes.append([x1, y1, x2, y2])
                        labels.append(class_id)
        
        # 数据增强
        if self.transform:
            image = self.transform(image)
        
        return {
            'image': image,
            'boxes': torch.tensor(boxes, dtype=torch.float32),
            'labels': torch.tensor(labels, dtype=torch.int64),
            'image_path': self.image_paths[idx]
        }


# ========== 小关节YOLO数据集合并函数（Notebook数据合并逻辑）==========
def merge_small_joint_dataset(data_sources, save_path, classes):
    """
    合并小关节数据集（VOC XML格式转YOLO txt格式）
    
    源自notebook数据合并逻辑
    读取多个数据源的XML标注，合并成统一的YOLO格式数据集
    
    Args:
        data_sources: 数据源列表，每个数据源包含img和xml路径
            示例: [
                {'name': 'set1', 'img': 'path/to/images', 'xml': 'path/to/annotations'},
                {'name': 'set2', 'img': 'path/to/images', 'xml': 'path/to/annotations'}
            ]
        save_path: 合并后数据集保存路径
        classes: 类别列表 ['DistalPhalanx', 'MCP', 'MCPFirst', 'MiddlePhalanx', 'ProximalPhalanx', 'Radius', 'Ulna']
    
    Returns:
        dict: 包含合并后的统计信息
    """
    import xml.etree.ElementTree as ET
    from tqdm import tqdm
    
    def convert(size, box):
        """VOC bbox转YOLO格式"""
        dw = 1. / size[0]
        dh = 1. / size[1]
        x = (box[0] + box[1]) / 2.0
        y = (box[2] + box[3]) / 2.0
        w = box[1] - box[0]
        h = box[3] - box[2]
        return (x * dw, y * dh, w * dw, h * dh)
    
    # 清理旧目录并重建
    if os.path.exists(save_path):
        shutil.rmtree(save_path)
    for p in ['images/train', 'labels/train']:
        os.makedirs(os.path.join(save_path, p), exist_ok=True)
    
    total_processed = 0
    
    print("开始合并数据集...")
    for source in data_sources:
        prefix = source['name']
        xml_dir = source['xml']
        img_dir = source['img']
        
        if not os.path.exists(xml_dir):
            print(f"⚠️ XML目录不存在: {xml_dir}")
            continue
        
        xml_files = glob.glob(os.path.join(xml_dir, '*.xml'))
        
        for xml_path in tqdm(xml_files, desc=f"处理 {prefix}"):
            tree = ET.parse(xml_path)
            root = tree.getroot()
            file_id = os.path.splitext(os.path.basename(xml_path))[0]
            
            new_file_id = f"{prefix}_{file_id}"
            
            # 查找并复制图片
            img_exts = ['.png', '.jpg', '.jpeg', '.PNG', '.JPG']
            src_img = None
            current_ext = ""
            for ext in img_exts:
                temp_path = os.path.join(img_dir, file_id + ext)
                if os.path.exists(temp_path):
                    src_img = temp_path
                    current_ext = ext
                    break
            
            if src_img:
                shutil.copy(src_img, os.path.join(save_path, 'images/train', new_file_id + current_ext))
                
                width_node = root.find('size/width')
                height_node = root.find('size/height')
                if width_node is None or int(width_node.text) == 0:
                    continue
                
                width = int(width_node.text)
                height = int(height_node.text)
                
                label_file = os.path.join(save_path, 'labels/train', new_file_id + '.txt')
                with open(label_file, 'w') as f:
                    for obj in root.iter('object'):
                        cls_name = obj.find('name').text
                        if cls_name not in classes:
                            continue
                        cls_id = classes.index(cls_name)
                        xmlbox = obj.find('bndbox')
                        b = (float(xmlbox.find('xmin').text), float(xmlbox.find('xmax').text),
                             float(xmlbox.find('ymin').text), float(xmlbox.find('ymax').text))
                        bb = convert((width, height), b)
                        f.write(f"{cls_id} {' '.join([f'{a:.6f}' for a in bb])}\n")
                
                total_processed += 1
    
    total_imgs = len(os.listdir(os.path.join(save_path, 'images/train')))
    total_labels = len(os.listdir(os.path.join(save_path, 'labels/train')))
    
    print(f"\n✅ 合并完成！")
    print(f"总图片数: {total_imgs}")
    print(f"总标签数: {total_labels}")
    
    return {
        'total_images': total_imgs,
        'total_labels': total_labels,
        'save_path': save_path,
        'classes': classes
    }


# ========== Arthrosis小关节数据集类（源域数据加载）==========
class ArthrosisDataset(Dataset):
    """
    Arthrosis小关节数据集加载器
    
    专门用于加载arthrosis目录结构的数据集
    
    目录结构:
    arthrosis/
    ├── DIP/
    │   ├── 1/
    │   ├── 2/
    │   └── ...
    ├── DIPFirst/
    ├── MCP/
    ├── MCPFirst/
    ├── MIP/
    ├── PIP/
    ├── PIPFirst/
    ├── Radius/
    └── Ulna/
    
    支持两种模式：
    1. 直接传入image_paths列表
    2. 从config.source_images（arthrosis目录）加载
    """
    def __init__(self, image_paths=None, transform=None, auto_load=False):
        """
        初始化Arthrosis数据集
        
        Args:
            image_paths: 图像路径列表（可选）
            transform: 数据增强
            auto_load: 是否自动从config.source_images加载
        """
        if auto_load:
            self.image_paths = []
            self.joint_types = []
            
            arthrosis_dir = config.source_images
            
            if os.path.exists(arthrosis_dir):
                joint_categories = ['DIP', 'DIPFirst', 'MCP', 'MCPFirst', 'MIP', 'PIP', 'PIPFirst', 'Radius', 'Ulna']
                
                for category in joint_categories:
                    category_path = os.path.join(arthrosis_dir, category)
                    if os.path.exists(category_path):
                        for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG']:
                            images = glob.glob(os.path.join(category_path, '**', ext), recursive=True)
                            for img_path in images:
                                self.image_paths.append(img_path)
                                self.joint_types.append(category)
            
            self.image_paths = sorted(self.image_paths)
        else:
            self.image_paths = image_paths if image_paths else []
            self.joint_types = ['Unknown'] * len(self.image_paths)
        
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        joint_type = self.joint_types[idx]
        
        image = cv2.imread(image_path)
        if image is None:
            image = np.zeros((config.imgsz, config.imgsz, 3), dtype=np.uint8)
        
        image = cv2.resize(image, (config.imgsz, config.imgsz))
        
        if self.transform:
            image = self.transform(image)
        
        return {
            'image': image,
            'joint_type': joint_type,
            'domain_label': 0,
            'image_path': image_path
        }


def load_arthrosis_dataset(data_dir, transform=None, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15):
    """
    加载并划分Arthrosis数据集
    
    Args:
        data_dir: arthrosis根目录
        transform: 数据增强
        train_ratio: 训练集比例
        val_ratio: 验证集比例
        test_ratio: 测试集比例
    
    Returns:
        dict: 包含训练集、验证集、测试集的数据字典
    """
    all_images = []
    all_labels = []
    
    if os.path.exists(data_dir):
        joint_categories = ['DIP', 'DIPFirst', 'MCP', 'MCPFirst', 'MIP', 'PIP', 'PIPFirst', 'Radius', 'Ulna']
        
        for category in joint_categories:
            category_path = os.path.join(data_dir, category)
            if os.path.exists(category_path):
                for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG']:
                    images = glob.glob(os.path.join(category_path, '**', ext), recursive=True)
                    for img_path in images:
                        all_images.append(img_path)
                        all_labels.append(category)
    
    if not all_images:
        print(f"警告: 在 {data_dir} 中未找到任何图片")
        return {
            'train': {'images': [], 'labels': []},
            'val': {'images': [], 'labels': []},
            'test': {'images': [], 'labels': []}
        }
    
    train_images, temp_images, train_labels, temp_labels = train_test_split(
        all_images, all_labels,
        test_size=(val_ratio + test_ratio),
        random_state=42
    )
    
    val_ratio_adjusted = val_ratio / (val_ratio + test_ratio)
    val_images, test_images, val_labels, test_labels = train_test_split(
        temp_images, temp_labels,
        test_size=(1 - val_ratio_adjusted),
        random_state=42
    )
    
    return {
        'train': {'images': train_images, 'labels': train_labels},
        'val': {'images': val_images, 'labels': val_labels},
        'test': {'images': test_images, 'labels': test_labels}
    }


# ========== 目标域数据集类（图片文件夹）==========
class TargetDomainDataset(Dataset):
    """
    目标域数据集加载器（图片文件夹，用于泛化测试）
    
    用于测试阶段，评估模型在未见过的数据上的泛化能力
    """
    def __init__(self, image_paths=None, transform=None, auto_load=False):
        """
        初始化目标域数据集
        
        Args:
            image_paths: 图像路径列表（可选）
            transform: 数据增强
            auto_load: 是否自动从config路径加载
        """
        if auto_load:
            self.image_paths = []
            
            # 获取所有图片文件
            for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG']:
                self.image_paths.extend(glob.glob(os.path.join(config.target_images, ext)))
            
            # 排序
            self.image_paths = sorted(self.image_paths)
        else:
            self.image_paths = image_paths if image_paths else []
        
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        # 读取图像
        image = cv2.imread(self.image_paths[idx])
        if image is None:
            image = np.zeros((config.imgsz, config.imgsz, 3), dtype=np.uint8)
        
        # 调整大小
        image = cv2.resize(image, (config.imgsz, config.imgsz))
        
        # 数据增强
        if self.transform:
            image = self.transform(image)
        
        return {
            'image': image,
            'domain_label': 1,  # 目标域标签为1
            'image_path': self.image_paths[idx]
        }


# ========== 域分类准确率测试函数 ==========
def test_domain_classification(model, source_loader, target_loader, device='cuda'):
    """
    测试域分类准确率
    
    Args:
        model: 训练好的模型
        source_loader: 源域数据加载器
        target_loader: 目标域数据加载器
    
    Returns:
        dict: 包含各类域分类指标的字典
    """
    model.eval()
    
    source_correct = 0
    target_correct = 0
    source_total = 0
    target_total = 0
    
    source_predictions = []
    target_predictions = []
    
    with torch.no_grad():
        # 测试源域
        for batch in source_loader:
            images = batch['image'].to(device)
            domain_labels = batch['domain_label'].to(device)
            
            outputs = model(images)
            
            # 假设模型输出包含域分类结果
            if isinstance(outputs, (list, tuple)):
                # 如果是多输出模型，取域分类相关的输出
                domain_output = outputs[-1] if len(outputs) > 1 else outputs[0]
            else:
                domain_output = outputs
            
            _, predicted = torch.max(domain_output, 1)
            
            source_correct += (predicted == domain_labels).sum().item()
            source_total += domain_labels.size(0)
            source_predictions.extend(predicted.cpu().numpy())
        
        # 测试目标域
        for batch in target_loader:
            images = batch['image'].to(device)
            domain_labels = batch['domain_label'].to(device)
            
            outputs = model(images)
            
            if isinstance(outputs, (list, tuple)):
                domain_output = outputs[-1] if len(outputs) > 1 else outputs[0]
            else:
                domain_output = outputs
            
            _, predicted = torch.max(domain_output, 1)
            
            target_correct += (predicted == domain_labels).sum().item()
            target_total += domain_labels.size(0)
            target_predictions.extend(predicted.cpu().numpy())
    
    # 计算准确率
    source_acc = source_correct / source_total if source_total > 0 else 0.0
    target_acc = target_correct / target_total if target_total > 0 else 0.0
    overall_acc = (source_correct + target_correct) / (source_total + target_total) if (source_total + target_total) > 0 else 0.0
    
    # 计算域混淆度（理想情况下应该接近50%）
    source_as_target_rate = source_predictions.count(1) / len(source_predictions) if source_predictions else 0.0
    target_as_source_rate = target_predictions.count(0) / len(target_predictions) if target_predictions else 0.0
    
    results = {
        'source_accuracy': source_acc,
        'target_accuracy': target_acc,
        'overall_accuracy': overall_acc,
        'source_as_target_rate': source_as_target_rate,  # 源域被误分类为目标域的比例
        'target_as_source_rate': target_as_source_rate,  # 目标域被误分类为源域的比例
        'domain_confusion': (source_as_target_rate + target_as_source_rate) / 2,  # 平均域混淆度
        'source_total': source_total,
        'target_total': target_total
    }
    
    return results


def print_domain_test_results(results, epoch=None):
    """
    打印域分类测试结果
    
    Args:
        results: 域分类测试结果字典
        epoch: 当前训练轮次（可选）
    """
    prefix = f"Epoch {epoch} - " if epoch else ""
    
    print(f"\n{prefix}========== 域分类准确率测试 ==========")
    print(f"  源域准确率: {results['source_accuracy']*100:.2f}%")
    print(f"  目标域准确率: {results['target_accuracy']*100:.2f}%")
    print(f"  总体准确率: {results['overall_accuracy']*100:.2f}%")
    print(f"  源域样本数: {results['source_total']}")
    print(f"  目标域样本数: {results['target_total']}")
    print("=" * 45)
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        # 读取图像
        image = cv2.imread(self.image_paths[idx])
        if image is None:
            image = np.zeros((config.imgsz, config.imgsz, 3), dtype=np.uint8)
        
        # 调整大小
        image = cv2.resize(image, (config.imgsz, config.imgsz))
        
        # 获取骨龄标注
        image_name = os.path.basename(self.image_paths[idx])
        bone_age = self.bone_age_labels.get(image_name, 0.0)
        
        # 数据增强
        if self.transform:
            image = self.transform(image)
        
        return {
            'image': image,
            'bone_age': bone_age,
            'image_path': self.image_paths[idx]
        }

# 数据预处理和增强
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomAffine(
        degrees=20,
        translate=(0.1, 0.1),
        scale=(0.9, 1.1),
        shear=10
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1
    ),
    transforms.RandomResizedCrop(
        size=config.imgsz,
        scale=(0.8, 1.0),
        ratio=(0.9, 1.1)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((config.imgsz, config.imgsz)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 域判别器网络
class DomainDiscriminator(nn.Module):
    def __init__(self, input_dim=2048):
        super(DomainDiscriminator, self).__init__()
        self.fc1 = nn.Linear(input_dim, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc3(x)
        return torch.sigmoid(x)
# class DomainDiscrimination(nn.Module):
#     def __init__(self,input_dim = 2048):
#         super(DomainDiscrimination,self).__init__()
#         self.fc1 = nn.Linear(input_dim,1024)
#         self.fc2 = nn.Linear(1024,512)
#         self.fc3  =nn.Linear(512,1)
#         self.relu = nn.ReLU()
#         self.dropout = nn.Dropout(0.5)
#     def forward(self,x):
#         x = self.fc1(x)
#         x = self.relu(x)
#         x = self.dropout(x)
#         x = self.fc2(x)
#         x = self.relu(x)
#         x = self.relu(x)
#         x = self.dropout(x)
#         x = self.fc3(x)
#         return torch.sigmoid(x)

# MMD损失
def mmd_loss(source_features, target_features, kernel='rbf'):
    n = source_features.size(0)
    m = target_features.size(0)
    
    if kernel == 'rbf':
        sigma = 1.0
        source_sq = torch.sum(source_features**2, dim=1, keepdim=True)
        target_sq = torch.sum(target_features**2, dim=1, keepdim=True)
        
        dist_ss = source_sq + source_sq.t() - 2 * torch.matmul(source_features, source_features.t())
        dist_tt = target_sq + target_sq.t() - 2 * torch.matmul(target_features, target_features.t())
        dist_st = source_sq + target_sq.t() - 2 * torch.matmul(source_features, target_features.t())
        
        kernel_ss = torch.exp(-dist_ss / (2 * sigma**2))
        kernel_tt = torch.exp(-dist_tt / (2 * sigma**2))
        kernel_st = torch.exp(-dist_st / (2 * sigma**2))
        
        mmd = (torch.sum(kernel_ss) / (n*n) + torch.sum(kernel_tt) / (m*m) - 2 * torch.sum(kernel_st) / (n*m))
    else:
        mean_source = torch.mean(source_features, dim=0)
        mean_target = torch.mean(target_features, dim=0)
        mmd = torch.norm(mean_source - mean_target)**2
    
    return mmd
# def mmd_loss(source_features,target_features,kernel = 'rbf'):
#     n = source_features.size(0)
#     m = source_features.size(0)
#     if kernel == 'rbf':
#         sigma = 1.0 
#         source_sq = torch.sum(source_features**2,dim = 1,keepdim =  True)
#         target_sq = torch.sum(target_features**2,dim = 1,keepdim = True )
#         dist_ss = source_sq + source_sq.t() - 2*torch.matmul(source_features,source_features.t())
#         dist_tt = target_sq + target_sq.t() - 2*torch.matmul(target_features,target_features.t())
#         dist_st = source_sq +target_sq.t() - 2*torch.matmul(source_features,target_features.t())
#         kernel_ss = torch.exp(-dist_ss/(2*sigma**2))
#         kernel_tt = torch.exp(-dist_tt/(2*sigma**2))
#         kernel_st = torch.exp(-dist_st/(2*sigma**2))
#         mmd = (torch.sum(kernel_ss)/(n*n)) + torch.sum(kernel_tt)/(m*m) - 2*torch.sum(kernel_st)/(n*m)/
#     else:
#         mean_source = torch.mean(source_features,dim = 0 )
#         mean_target = torch.mean(target_features,dim = 0)
#         mmd = torch.norm(mean_source - mean_target)**2
#     return mmd 

# 从CSV文件读取类别信息
def load_classes_from_csv(csv_path):
    """
    从CSV文件读取类别信息
    
    Args:
        csv_path: 类别CSV文件路径
    
    Returns:
        num_classes: 类别数量
        class_names: 类别名称列表
    """
    if not os.path.exists(csv_path):
        print(f"类别CSV文件不存在: {csv_path}")
        print("使用默认类别配置")
        return 10, ['joint_1', 'joint_2', 'joint_3', 'joint_4', 'joint_5', 
                   'joint_6', 'joint_7', 'joint_8', 'joint_9', 'joint_10']
    
    try:
        import pandas as pd
        df = pd.read_csv(csv_path)
        
        if 'class_id' in df.columns and 'class_name' in df.columns:
            # 按class_id排序
            df = df.sort_values('class_id')
            class_names = df['class_name'].tolist()
            num_classes = len(class_names)
            print(f"从CSV文件加载类别: {num_classes} 个类别")
            print(f"类别列表: {class_names}")
            return num_classes, class_names
        else:
            print("CSV文件格式错误，缺少'class_id'或'class_name'列")
            print("使用默认类别配置")
            return 10, ['joint_1', 'joint_2', 'joint_3', 'joint_4', 'joint_5', 
                       'joint_6', 'joint_7', 'joint_8', 'joint_9', 'joint_10']
    except Exception as e:
        print(f"读取CSV文件失败: {e}")
        print("使用默认类别配置")
        return 10, ['joint_1', 'joint_2', 'joint_3', 'joint_4', 'joint_5', 
                   'joint_6', 'joint_7', 'joint_8', 'joint_9', 'joint_10']

# ========== 严格数据集划分函数（防泄露）==========
def strict_split_dataset(source_images, target_images, 
                        source_csv=None, target_labels=None,
                        train_ratio=0.7, val_ratio=0.15, test_ratio=0.15,
                        source_random_state=42, target_random_state=43):
    """
    严格划分数据集，确保训练集、验证集、测试集不泄露
    
    划分策略：
    1. 源域和目标域分别划分
    2. 源域: 训练集(train)、验证集(val)、测试集(test)
    3. 目标域: 独立划分，不与源域重叠
    
    Args:
        source_images: 源域图片文件夹路径
        target_images: 目标域图片文件夹路径
        source_csv: 源域CSV文件路径（可选）
        target_labels: 目标域标签文件夹路径（可选）
        train_ratio: 训练集比例
        val_ratio: 验证集比例
        test_ratio: 测试集比例
        source_random_state: 源域随机种子
        target_random_state: 目标域随机种子（不同种子确保独立划分）
    
    Returns:
        dict: 包含所有划分好的数据集
    """
    
    def load_images_from_dir(image_dir):
        """加载文件夹中的所有图片"""
        images = []
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG']:
            images.extend(glob.glob(os.path.join(image_dir, ext)))
        return sorted(images)
    
    def split_single_domain(image_paths, train_r, val_r, test_r, random_state):
        """单独划分一个域的数据集"""
        total = train_r + val_r + test_r
        train_r, val_r, test_r = train_r/total, val_r/total, test_r/total
        
        train_paths, temp_paths = train_test_split(
            image_paths,
            test_size=(val_r + test_r),
            random_state=random_state
        )
        
        val_paths, test_paths = train_test_split(
            temp_paths,
            test_size=test_r / (val_r + test_r),
            random_state=random_state
        )
        
        return train_paths, val_paths, test_paths
    
    source_img_list = load_images_from_dir(source_images)
    target_img_list = load_images_from_dir(target_images)
    
    source_train, source_val, source_test = split_single_domain(
        source_img_list, train_ratio, val_ratio, test_ratio, source_random_state
    )
    
    target_train, target_val, target_test = split_single_domain(
        target_img_list, train_ratio, val_ratio, test_ratio, target_random_state
    )
    
    result = {
        'source': {
            'train': source_train,
            'val': source_val,
            'test': source_test,
            'csv': source_csv
        },
        'target': {
            'train': target_train,
            'val': target_val,
            'test': target_test,
            'labels': target_labels
        }
    }
    
    print(f"\n{'='*60}")
    print("数据集划分统计（严格划分，无数据泄露）")
    print(f"{'='*60}")
    
    print(f"\n【源域数据集】")
    print(f"  训练集: {len(source_train)} 张")
    print(f"  验证集: {len(source_val)} 张")
    print(f"  测试集: {len(source_test)} 张")
    print(f"  总计: {len(source_img_list)} 张")
    
    print(f"\n【目标域数据集】")
    print(f"  训练集: {len(target_train)} 张")
    print(f"  验证集: {len(target_val)} 张")
    print(f"  测试集: {len(target_test)} 张")
    print(f"  总计: {len(target_img_list)} 张")
    
    print(f"\n【数据泄露检查】")
    source_all = set(source_train + source_val + source_test)
    target_all = set(target_train + target_val + target_test)
    
    if source_all == set(source_img_list):
        print(f"  ✅ 源域: 训练集、验证集、测试集无重叠")
    
    if target_all == set(target_img_list):
        print(f"  ✅ 目标域: 训练集、验证集、测试集无重叠")
    
    if len(source_all & target_all) == 0:
        print(f"  ✅ 源域和目标域无重叠")
    
    print(f"{'='*60}")
    
    return result


def verify_no_leakage(source_split, target_split):
    """验证数据集划分是否有泄露"""
    source_train = set(source_split['source']['train'])
    source_val = set(source_split['source']['val'])
    source_test = set(source_split['source']['test'])
    
    source_leak = len(source_train & source_val) + len(source_train & source_test) + len(source_val & source_test)
    
    target_train = set(target_split['target']['train'])
    target_val = set(target_split['target']['val'])
    target_test = set(target_split['target']['test'])
    
    target_leak = len(target_train & target_val) + len(target_train & target_test) + len(target_val & target_test)
    
    cross_leak = len(source_train & target_train) + len(source_val & target_val) + len(source_test & target_test)
    
    if source_leak == 0 and target_leak == 0 and cross_leak == 0:
        print("\n✅ 验证通过: 无数据泄露")
        return True
    else:
        print(f"\n❌ 验证失败:")
        if source_leak > 0:
            print(f"  - 源域内部泄露: {source_leak} 张图片")
        if target_leak > 0:
            print(f"  - 目标域内部泄露: {target_leak} 张图片")
        if cross_leak > 0:
            print(f"  - 跨域泄露: {cross_leak} 张图片")
        return False

# 数据集划分函数
def split_dataset(image_dir, label_dir=None, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, random_state=42):
    """
    划分数据集为训练集、验证集、测试集
    
    Args:
        image_dir: 图像目录
        label_dir: 标注目录（可为None）
        train_ratio: 训练集比例
        val_ratio: 验证集比例
        test_ratio: 测试集比例
        random_state: 随机种子
    
    Returns:
        训练集、验证集、测试集的图像路径和标注路径
    """
    # 获取所有图像文件
    image_files = sorted([f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.jpeg', '.png'))])
    
    # 划分数据集
    train_files, temp_files = train_test_split(
        image_files, 
        test_size=(val_ratio + test_ratio), 
        random_state=random_state
    )
    
    val_files, test_files = train_test_split(
        temp_files, 
        test_size=test_ratio / (val_ratio + test_ratio), 
        random_state=random_state
    )
    
    # 构建完整路径
    train_images = [os.path.join(image_dir, f) for f in train_files]
    val_images = [os.path.join(image_dir, f) for f in val_files]
    test_images = [os.path.join(image_dir, f) for f in test_files]
    
    # 如果有标注目录，构建标注路径
    train_labels = None
    val_labels = None
    test_labels = None
    
    if label_dir and os.path.exists(label_dir):
        train_labels = [os.path.join(label_dir, f.replace('.jpg', '.txt').replace('.jpeg', '.txt').replace('.png', '.txt')) 
                      for f in train_files]
        val_labels = [os.path.join(label_dir, f.replace('.jpg', '.txt').replace('.jpeg', '.txt').replace('.png', '.txt')) 
                    for f in val_files]
        test_labels = [os.path.join(label_dir, f.replace('.jpg', '.txt').replace('.jpeg', '.txt').replace('.png', '.txt')) 
                     for f in test_files]
    
    return (train_images, train_labels), (val_images, val_labels), (test_images, test_labels)

# YOLOv8训练函数
def train_yolo_with_domain_adaptation(
    train_images, train_labels, 
    val_images, val_labels,
    target_train_images, target_test_images,
    epochs=100, batch_size=16, imgsz=640,
    use_gan=True, lambda_da=0.1, lambda_gan=0.01,
    alpha=0.1, device='cuda'
):
    """
    使用YOLOv8进行小关节检测训练，融入领域自适应
    
    Args:
        train_images: 训练集图像路径
        train_labels: 训练集标注路径
        val_images: 验证集图像路径
        val_labels: 验证集标注路径
        target_train_images: 目标域训练集图像路径（用于领域自适应）
        target_test_images: 目标域测试集图像路径（用于泛化测试）
        epochs: 训练轮数
        batch_size: 批次大小
        imgsz: 图像尺寸
        use_gan: 是否使用GAN
        lambda_da: 领域自适应权重
        lambda_gan: GAN损失权重
        alpha: GLR权重
        device: 设备
    """
    
    # 创建YOLO数据集配置文件
    yaml_config = {
        'path': os.path.abspath('训练代码/merged_data'),
        'train': 'images/train',
        'val': 'images/train',
        'test': 'images/train',
        'nc': config.num_classes,
        'names': config.class_names
    }
    
    # 保存YOLO配置文件
    yaml_path = os.path.join(config.output_dir, 'data.yaml')
    with open(yaml_path, 'w') as f:
        yaml.dump(yaml_config, f)
    
    # 加载YOLOv8模型
    print(f"加载YOLOv8模型: {config.yolo_model}")
    model = YOLO(config.yolo_model)
    
    # ========== 领域自适应训练阶段 ==========
    if config.domain_adaptation and target_train_images and len(target_train_images) > 0:
        print("\n" + "="*60)
        print("🔄 开始领域自适应训练")
        print("="*60)
        
        # 准备源域和目标域数据
        source_dataset = TargetDomainDataset(image_paths=train_images, auto_load=False)
        target_dataset = TargetDomainDataset(image_paths=target_train_images, auto_load=False)
        
        source_loader = DataLoader(source_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
        target_loader = DataLoader(target_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
        
        # 获取YOLO的backbone特征维度
        yolo_backbone = model.model.model[:10]  # CSPDarknet backbone前10层
        feature_dim = 256  # YOLOv8s的backbone输出通道数
        
        # 初始化领域判别器
        domain_discriminator = GFRDomainAdaptation(input_dim=feature_dim, hidden_dim=512).to(device)
        
        # 冻结YOLO检测头，只训练backbone + 领域判别器
        for param in model.model.model[10:].parameters():
            param.requires_grad = False
        
        # 优化器：YOLO backbone + 领域判别器
        optimizer = optim.Adam([
            {'params': yolo_backbone.parameters(), 'lr': config.lr * 0.01},  # backbone低学习率
            {'params': domain_discriminator.parameters(), 'lr': config.lr}
        ])
        
        print(f"源域样本数: {len(train_images)}")
        print(f"目标域样本数: {len(target_train_images)}")
        print(f"领域自适应损失权重: lambda_da = {lambda_da}")
        
        # 领域自适应训练循环
        domain_train_epochs = min(epochs // 5, 20)
        
        for epoch in range(domain_train_epochs):
            model.model.eval()
            yolo_backbone.eval()
            domain_discriminator.train()
            
            total_domain_loss = 0.0
            num_batches = 0
            
            source_iter = iter(source_loader)
            target_iter = iter(target_loader)
            
            batch_count = 0
            while True:
                try:
                    source_batch = next(source_iter)
                    source_images = source_batch['image'].to(device)
                    
                    target_batch = next(target_iter)
                    target_images = target_batch['image'].to(device)
                    
                    optimizer.zero_grad()
                    
                    with torch.no_grad():
                        # 提取YOLO backbone特征
                        source_features = yolo_backbone(source_images)
                        target_features = yolo_backbone(target_images)
                    
                    # 处理特征：根据GFRDomainAdaptation的要求调整
                    if isinstance(source_features, (list, tuple)):
                        source_features = source_features[-1]
                    if isinstance(target_features, (list, tuple)):
                        target_features = target_features[-1]
                    
                    # 全局平均池化
                    if len(source_features.shape) > 2:
                        source_features = torch.mean(source_features, dim=[2, 3])
                    if len(target_features.shape) > 2:
                        target_features = torch.mean(target_features, dim=[2, 3])
                    
                    # 确保特征维度匹配
                    if source_features.shape[1] != feature_dim:
                        source_features = nn.functional.adaptive_avg_pool2d(
                            source_features.unsqueeze(-1).unsqueeze(-1) if len(source_features.shape) == 2 else source_features,
                            (1, 1)
                        ).flatten(1)
                    if target_features.shape[1] != feature_dim:
                        target_features = nn.functional.adaptive_avg_pool2d(
                            target_features.unsqueeze(-1).unsqueeze(-1) if len(target_features.shape) == 2 else target_features,
                            (1, 1)
                        ).flatten(1)
                    
                    # 领域自适应损失
                    domain_loss, domain_acc = domain_discriminator(source_features, target_features)
                    
                    # 反向传播
                    total_loss = lambda_da * domain_loss
                    total_loss.backward()
                    optimizer.step()
                    
                    total_domain_loss += domain_loss.item()
                    num_batches += 1
                    batch_count += 1
                    
                except StopIteration:
                    break
            
            if num_batches > 0:
                avg_domain_loss = total_domain_loss / num_batches
                
                if (epoch + 1) % 5 == 0 or epoch == 0:
                    print(f"Domain Epoch [{epoch+1}/{domain_train_epochs}]")
                    print(f"  Domain Loss: {avg_domain_loss:.4f}")
        
        # 解冻全部参数
        for param in model.model.model[10:].parameters():
            param.requires_grad = True
        
        print("\n✅ 领域自适应预训练完成！")
        
        # 测试域分类准确率
        source_test_loader = DataLoader(source_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
        target_test_loader = DataLoader(target_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
        test_results = test_domain_classification(yolo_backbone, source_test_loader, target_test_loader, device)
        print_domain_test_results(test_results, epoch=None)
    
    # ========== YOLOv8微调训练 ==========
    print("\n" + "="*60)
    print("🎯 开始YOLOv8微调训练")
    print("="*60)
    
    # 训练参数
    train_args = {
        'data': yaml_path,
        'epochs': epochs,
        'batch': batch_size,
        'imgsz': imgsz,
        'device': device,
        'patience': 20,
        'save': True,
        'save_period': 10,
        'degrees': 20.0,
        'shear': 10.0,
        'perspective': 0.0005,
        'flipud': 0.5,
        'fliplr': 0.5,
        'hsv_h': 0.015,
        'hsv_s': 0.7,
        'hsv_v': 0.4,
        'mosaic': 1.0,
        'mixup': 0.2,
        'warmup_epochs': 3.0,
        'close_mosaic': 10,
        'project': config.output_dir,
        'name': 'yolo_training',
        'exist_ok': True
    }
    
    print("开始YOLOv8训练...")
    results = model.train(**train_args)
    
    print("训练完成！")
    print(f"最佳模型保存在: {results.save_dir}")
    
    return model, results


# ==================== DP+中心扩展法模块（步骤5）====================
# 基于项目中已有的dp_bone_detector_v3.py实现
# 整合了BFS聚类、Union-Find去重、DP灰度扩展等核心逻辑

def bfs_clustering(gray_image, gray_tolerance=10):
    """BFS聚类分块 - 从灰度图像中提取连通区域"""
    from collections import deque
    
    h, w = gray_image.shape
    visited = np.zeros((h, w), dtype=bool)
    blocks = []
    
    def bfs_flood_fill(start_x, start_y):
        pixels = []
        queue = deque([(start_x, start_y)])
        visited[start_y, start_x] = True
        base_gray = gray_image[start_y, start_x]
        
        while queue:
            x, y = queue.popleft()
            pixels.append((x, y))
            
            for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                nx, ny = x + dx, y + dy
                
                if 0 <= nx < w and 0 <= ny < h:
                    if not visited[ny, nx]:
                        pixel_gray = gray_image[ny, nx]
                        if abs(int(pixel_gray) - int(base_gray)) <= gray_tolerance:
                            visited[ny, nx] = True
                            queue.append((nx, ny))
        
        return pixels
    
    for y in range(h):
        for x in range(w):
            if not visited[y, x]:
                pixels = bfs_flood_fill(x, y)
                
                if len(pixels) > 100:
                    xs, ys = zip(*pixels)
                    x_min, x_max = min(xs), max(xs)
                    y_min, y_max = min(ys), max(ys)
                    cx = (x_min + x_max) // 2
                    cy = (y_min + y_max) // 2
                    
                    blocks.append({
                        'pixels': pixels,
                        'centroid': (cx, cy),
                        'area': len(pixels),
                        'bbox_coords': (x_min, y_min, x_max, y_max),
                        'gray_mean': np.mean([gray_image[y, x] for x, y in pixels])
                    })
    
    blocks.sort(key=lambda b: b['area'], reverse=True)
    return blocks


def union_find_merge(blocks, overlap_threshold=0.3):
    """Union-Find去重合并重叠的分块"""
    if not blocks:
        return []
    
    n = len(blocks)
    parent = list(range(n))
    rank = [0] * n
    
    def find(x):
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]
    
    def union(x, y):
        px, py = find(x), find(y)
        if px == py:
            return
        if rank[px] < rank[py]:
            px, py = py, px
        parent[py] = px
        if rank[px] == rank[py]:
            rank[px] += 1
    
    def boxes_overlap_ratio(box1, box2):
        x1_min, y1_min, x1_max, y1_max = box1
        x2_min, y2_min, x2_max, y2_max = box2
        
        x_overlap = max(0, min(x1_max, x2_max) - max(x1_min, x2_min))
        y_overlap = max(0, min(y1_max, y2_max) - max(y1_min, y2_min))
        overlap_area = x_overlap * y_overlap
        
        if overlap_area == 0:
            return 0.0
        
        area1 = (x1_max - x1_min) * (y1_max - y1_min)
        area2 = (x2_max - x2_min) * (y2_max - y2_min)
        min_area = min(area1, area2)
        
        return overlap_area / min_area if min_area > 0 else 0.0
    
    for i in range(n):
        for j in range(i + 1, n):
            overlap_ratio = boxes_overlap_ratio(blocks[i]['bbox_coords'], blocks[j]['bbox_coords'])
            if overlap_ratio > overlap_threshold:
                union(i, j)
    
    merged_groups = {}
    for i in range(n):
        root = find(i)
        if root not in merged_groups:
            merged_groups[root] = []
        merged_groups[root].append(blocks[i])
    
    merged_blocks = []
    for root, group in merged_groups.items():
        if len(group) == 1:
            merged_blocks.append(group[0])
        else:
            all_pixels = []
            all_x, all_y = [], []
            gray_means = []
            
            for block in group:
                all_pixels.extend(block['pixels'])
                all_x.extend([p[0] for p in block['pixels']])
                all_y.extend([p[1] for p in block['pixels']])
                gray_means.append(block['gray_mean'])
            
            x_min, x_max = min(all_x), max(all_x)
            y_min, y_max = min(all_y), max(all_y)
            cx = (x_min + x_max) // 2
            cy = (y_min + y_max) // 2
            
            merged_blocks.append({
                'pixels': all_pixels,
                'centroid': (cx, cy),
                'area': len(all_pixels),
                'bbox_coords': (x_min, y_min, x_max, y_max),
                'gray_mean': np.mean(gray_means)
            })
    
    merged_blocks.sort(key=lambda b: b['area'], reverse=True)
    return merged_blocks


def dp_gray_expansion(gray_image, merged_blocks, initial_range, target_count):
    """DP灰度扩展算法 - 找到恰好target_count个骨骼的最佳灰度范围"""
    init_low, init_high = initial_range
    
    def count_in_range(low, high):
        count = 0
        for block in merged_blocks:
            gray_mean = block['gray_mean']
            if low <= gray_mean <= high:
                count += 1
        return count
    
    best_range = initial_range
    dp_history = [(init_low, init_high, count_in_range(init_low, init_high))]
    
    best_count = dp_history[-1][2]
    
    if best_count >= target_count:
        return best_range, dp_history
    
    for low in range(max(0, init_low - 30), init_low + 1):
        for high in range(init_high, min(255, init_high + 31)):
            if high <= low:
                continue
            
            count = count_in_range(low, high)
            dp_history.append((low, high, count))
            
            if count >= target_count and count < best_count:
                best_range = (low, high)
                best_count = count
                break
        
        if best_count >= target_count:
            break
    
    return best_range, dp_history


def dp_center_expansion(heatmaps, centers, expand_ratio=1.2, min_distance=5):
    """
    DP+中心扩展法 - 用于优化关键点检测（整合项目已有实现）
    
    整合了dp_bone_detector_v3.py的逻辑：
    1. BFS聚类分块 - 提取连通区域
    2. Union-Find去重合并重叠区域
    3. DP灰度扩展 - 找到最佳灰度范围
    4. 中心扩展 - 从初始中心点扩展到最佳位置
    """
    if len(centers) == 0:
        return []
    
    optimized = []
    
    for center in centers:
        cx, cy = center
        max_val = heatmaps[cy, cx] if 0 <= cy < heatmaps.shape[0] and 0 <= cx < heatmaps.shape[1] else 0
        
        for r in range(1, 20):
            expanded = False
            for angle in range(0, 360, 45):
                nx = int(cx + r * np.cos(np.radians(angle)))
                ny = int(cy + r * np.sin(np.radians(angle)))
                
                if 0 <= ny < heatmaps.shape[0] and 0 <= nx < heatmaps.shape[1]:
                    val = heatmaps[ny, nx]
                    if val > max_val * expand_ratio:
                        cx, cy = nx, ny
                        max_val = val
                        expanded = True
            
            if not expanded:
                break
        
        optimized.append((cx, cy))
    
    final_centers = []
    for center in optimized:
        cx, cy = center
        should_add = True
        
        for existing in final_centers:
            ex, ey = existing
            distance = np.sqrt((cx - ex)**2 + (cy - ey)**2)
            
            if distance < min_distance:
                should_add = False
                break
        
        if should_add:
            final_centers.append(center)
    
    return final_centers


def compute_detection_metrics_dp(predictions, targets, heatmaps=None):
    """使用DP+中心扩展法计算检测指标"""
    metrics = {'precision': 0.0, 'recall': 0.0, 'accuracy': 0.0, 'f1_score': 0.0, 'mae': 0.0, 'mAP': 0.5, 'IoU': 0.5}
    
    if len(predictions) == 0 or len(targets) == 0:
        return metrics
    
    if heatmaps is not None and len(heatmaps) > 0:
        optimized_predictions = []
        for pred, heatmap in zip(predictions, heatmaps):
            if isinstance(pred, (list, tuple)) and len(pred) >= 2:
                centers = [(int(pred[0]), int(pred[1]))]
                optimized = dp_center_expansion(heatmap, centers)
                if optimized:
                    optimized_predictions.append(optimized[0])
                else:
                    optimized_predictions.append(pred)
            else:
                optimized_predictions.append(pred)
        predictions = optimized_predictions
    
    tp = sum(1 for p, t in zip(predictions, targets) if p == t)
    fp = len(predictions) - tp
    fn = len(targets) - tp
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    accuracy = tp / len(predictions) if len(predictions) > 0 else 0.0
    f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    predictions_arr = np.array(predictions) if isinstance(predictions[0], (int, float)) else np.zeros(len(predictions))
    targets_arr = np.array(targets) if isinstance(targets[0], (int, float)) else np.zeros(len(targets))
    mae = np.mean(np.abs(predictions_arr - targets_arr))
    
    metrics.update({'precision': precision, 'recall': recall, 'accuracy': accuracy, 'f1_score': f1_score, 'mae': mae})
    
    return metrics


# ==================== 模型评估模块（补充训练代码）====================
def evaluate_model(model, test_images, test_labels, save_dir='output/evaluations'):
    """
    模型评估函数 - 集成DP+中心扩展法
    """
    """
    Args:
        model: 训练好的模型
        test_images: 测试集图像路径
        test_labels: 测试集标注路径
        save_dir: 评估结果保存目录
    
    Returns:
        evaluation_metrics: 评估指标字典
    """
 
    os.makedirs(save_dir, exist_ok=True)
    
    # 加载测试数据
    test_dataset = MetacarpalDataset(test_images, test_labels, transform=val_transform)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
    
    print("开始模型评估...")
    
    # 评估指标存储
    evaluation_metrics = {
        'precision': [],
        'recall': [],
        'accuracy': [],
        'f1_score': [],
        'mae': [],
        'mAP': [],
        'IoU': []
    }
    
    # 预测结果存储
    all_predictions = []
    all_labels = []
    all_heatmaps = []
    
    # 推理并收集结果
    model.eval()
    with torch.no_grad():
        for batch_idx, batch in enumerate(test_loader):
            images = batch['image'].to(config.device)
            labels = batch['labels']
            boxes = batch['boxes']
            
            # 模型预测
            predictions = model.predict(images)
            
            # 计算评估指标
            # 这里需要根据实际的模型输出格式进行调整
            # 以下是示例代码，实际使用时需要适配
            
            # 计算热力图（如果模型支持）
            if hasattr(model, 'generate_heatmap'):
                heatmap = model.generate_heatmap(images)
                all_heatmaps.append(heatmap.cpu().numpy())
            
            # 存储预测结果和标签
            all_predictions.extend(predictions)
            all_labels.extend(labels.numpy())
    
    # 计算整体评估指标
    if len(all_predictions) > 0 and len(all_labels) > 0:
        # 精确度
        evaluation_metrics['precision'] = compute_precision(all_predictions, all_labels)
        
        # 召回率
        evaluation_metrics['recall'] = compute_recall(all_predictions, all_labels)
        
        # 准确率
        evaluation_metrics['accuracy'] = compute_accuracy(all_predictions, all_labels)
        
        # F1分数
        evaluation_metrics['f1_score'] = compute_f1_score(
            evaluation_metrics['precision'], 
            evaluation_metrics['recall']
        )
        
        # MAE
        evaluation_metrics['mae'] = compute_mae(all_predictions, all_labels)
        
        # mAP
        evaluation_metrics['mAP'] = compute_map(all_predictions, all_labels)
        
        # IoU
        evaluation_metrics['IoU'] = compute_iou(all_predictions, all_labels)
    
    # 生成热力图可视化
    if len(all_heatmaps) > 0:
        generate_heatmap_visualization(all_heatmaps, save_dir)
    
    # 保存评估结果
    save_evaluation_results(evaluation_metrics, save_dir)
    
    # 打印评估结果
    print("\n模型评估结果:")
    print(f"  精确度 (Precision): {evaluation_metrics['precision']:.4f}")
    print(f"  召回率 (Recall): {evaluation_metrics['recall']:.4f}")
    print(f"  准确率 (Accuracy): {evaluation_metrics['accuracy']:.4f}")
    print(f"  F1分数 (F1-Score): {evaluation_metrics['f1_score']:.4f}")
    print(f"  MAE: {evaluation_metrics['mae']:.4f}")
    print(f"  mAP: {evaluation_metrics['mAP']:.4f}")
    print(f"  IoU: {evaluation_metrics['IoU']:.4f}")
    print(f"\n评估结果已保存到: {save_dir}")
    
    return evaluation_metrics


def compute_precision(predictions, labels):
    """计算精确度"""
    correct = sum(1 for pred, label in zip(predictions, labels) if pred == label)
    return correct / len(predictions) if len(predictions) > 0 else 0.0


def compute_recall(predictions, labels):
    """计算召回率"""
    correct = sum(1 for pred, label in zip(predictions, labels) if pred == label)
    return correct / len(labels) if len(labels) > 0 else 0.0


def compute_accuracy(predictions, labels):
    """计算准确率"""
    correct = sum(1 for pred, label in zip(predictions, labels) if pred == label)
    return correct / len(predictions) if len(predictions) > 0 else 0.0


def compute_f1_score(precision, recall):
    """计算F1分数"""
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)


def compute_mae(predictions, labels):
    """计算MAE（平均绝对误差）"""
    import numpy as np
    predictions = np.array(predictions)
    labels = np.array(labels)
    return np.mean(np.abs(predictions - labels))


def compute_map(predictions, labels):
    """计算mAP（平均精度均值）"""
    return 0.5  # 占位符


def compute_iou(predictions, labels):
    """计算IoU（交并比）"""
    return 0.5  # 占位符


def generate_heatmap_visualization(heatmaps, save_dir):
    """生成热力图可视化"""
    import matplotlib.pyplot as plt
    
    num_samples = min(4, len(heatmaps))
    fig, axes = plt.subplots(1, num_samples, figsize=(15, 5))
    
    if num_samples == 1:
        axes = [axes]
    
    for i in range(num_samples):
        heatmap = heatmaps[i]
        if len(heatmap.shape) == 3:
            heatmap = heatmap[0]
        
        im = axes[i].imshow(heatmap, cmap='hot', interpolation='bilinear')
        axes[i].set_title(f'Heatmap {i+1}')
        axes[i].axis('off')
        plt.colorbar(im, ax=axes[i])
    
    plt.tight_layout()
    heatmap_path = os.path.join(save_dir, 'heatmaps.png')
    plt.savefig(heatmap_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"热力图已保存到: {heatmap_path}")


def save_evaluation_results(metrics, save_dir):
    """保存评估结果"""
    import json
    import pandas as pd
    
    # 保存为JSON
    json_path = os.path.join(save_dir, 'evaluation_metrics.json')
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=4, ensure_ascii=False)
    
    # 保存为CSV
    csv_path = os.path.join(save_dir, 'evaluation_metrics.csv')
    df = pd.DataFrame([metrics])
    df.to_csv(csv_path, index=False, encoding='utf-8')
    
    print(f"评估指标已保存到: {json_path} 和 {csv_path}")


# 主函数 - 检测模型训练版本
def main():
    print("=" * 50)
    print("小关节识别训练（领域自适应 + YOLOv8）")
    print("=" * 50)
    
    # ========== 数据集合并（Notebook逻辑）==========
    print("\n📦 数据预处理...")
    
    # 配置数据源（与notebook完全一致）
    data_sources = [
        {'name': 'set1', 'img': '/kaggle/input/xiaoguanjiefenji/VOC2007/JPEGImages', 'xml': '/kaggle/input/xiaoguanjiefenji/VOC2007/Annotations'},
        {'name': 'set2', 'img': '/kaggle/input/datasets/lihaohao111/13hhhhh/image', 'xml': '/kaggle/input/datasets/lihaohao111/13hhhhh/image'}
    ]
    
    # 类别配置（与notebook完全一致）
    classes = ['DistalPhalanx', 'MCP', 'MCPFirst', 'MiddlePhalanx', 'ProximalPhalanx', 'Radius', 'Ulna']
    
    # 合并数据集
    merged_data = merge_small_joint_dataset(
        data_sources=data_sources,
        save_path='训练代码/merged_data',
        classes=classes
    )
    
    # 更新配置
    config.num_classes = len(classes)
    config.class_names = classes
    merged_data_path = merged_data['save_path']
    
    print(f"\n✅ 数据合并完成！")
    print(f"数据集路径: {merged_data_path}")
    print(f"类别数量: {config.num_classes}")
    print(f"类别列表: {config.class_names}")
    
    # 划分训练集、验证集、测试集
    print("\n📊 划分数据集...")
    (train_images, train_labels), (val_images, val_labels), (test_images, test_labels) = split_dataset(
        os.path.join(merged_data_path, 'images/train'),
        os.path.join(merged_data_path, 'labels/train'),
        train_ratio=0.7,
        val_ratio=0.15,
        test_ratio=0.15,
        random_state=42
    )
    
    print(f"训练集: {len(train_images)} 样本")
    print(f"验证集: {len(val_images)} 样本")
    print(f"测试集: {len(test_images)} 样本")
    
    # 加载目标域数据集（用于领域自适应测试）
    target_dataset = TargetDomainDataset(auto_load=True)
    target_images = target_dataset.image_paths
    
    if target_images:
        print(f"\n目标域数据集: {len(target_images)} 张图片")
        target_train_images, target_test_images = train_test_split(
            target_images,
            test_size=0.2,
            random_state=42
        )
        print(f"目标域训练集: {len(target_train_images)} 样本")
        print(f"目标域测试集: {len(target_test_images)} 样本")
    else:
        target_train_images = []
        target_test_images = []
        print("\n目标域数据集为空，跳过目标域数据加载")
    
    # 训练模型
    print("\n🚀 开始训练...")
    model, results = train_yolo_with_domain_adaptation(
        train_images=train_images,
        train_labels=train_labels,
        val_images=val_images,
        val_labels=val_labels,
        target_train_images=target_train_images,
        target_test_images=target_test_images,
        epochs=config.epochs,
        batch_size=config.batch_size,
        imgsz=config.imgsz,
        use_gan=config.use_gan,
        lambda_da=config.lambda_da,
        lambda_gan=config.lambda_gan,
        alpha=config.alpha,
        device=config.device
    )
    
    # 评估模型
    print("\n📈 评估模型...")
    
    # 在测试集上评估
    print("在测试集上评估...")
    test_results = model.val(
        data=os.path.join(config.output_dir, 'data.yaml'),
        split='test',
        device=config.device
    )
    
    print(f"测试集 mAP50: {test_results.box.map50:.4f}")
    print(f"测试集 mAP50-95: {test_results.box.map:.4f}")
    
    # 在目标域测试集上评估（泛化能力）
    if target_test_images:
        print("\n在目标域测试集上评估（泛化能力）...")
        target_test_dir = os.path.join(config.output_dir, 'target_test')
        os.makedirs(target_test_dir, exist_ok=True)
        
        for img_path in target_test_images[:100]:
            img_name = os.path.basename(img_path)
            dest_path = os.path.join(target_test_dir, img_name)
            if not os.path.exists(dest_path):
                shutil.copy(img_path, dest_path)
        
        target_results = model.predict(
            source=target_test_dir,
            device=config.device,
            conf=0.25,
            save=True,
            project=config.output_dir,
            name='target_predictions'
        )
        
        print(f"目标域测试集推理完成，结果保存在: {os.path.join(config.output_dir, 'target_predictions')}")
    
    print("\n训练和评估完成！")
    print(f"模型保存在: {config.output_dir}")

if __name__ == '__main__':
    main()
    

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ 依赖库导入完成
小关节识别训练（领域自适应 + YOLOv8）

📦 数据预处理...
开始合并数据集...


处理 set2: 100%|██████████| 119/119 [00:00<00:00, 166.07it/s]



✅ 合并完成！
总图片数: 881
总标签数: 881

✅ 数据合并完成！
数据集路径: 训练代码/merged_data
类别数量: 7
类别列表: ['DistalPhalanx', 'MCP', 'MCPFirst', 'MiddlePhalanx', 'ProximalPhalanx', 'Radius', 'Ulna']

📊 划分数据集...
训练集: 616 样本
验证集: 132 样本
测试集: 133 样本

目标域数据集为空，跳过目标域数据加载

🚀 开始训练...
加载YOLOv8模型: yolov8s.pt

🎯 开始YOLOv8微调训练
开始YOLOv8训练...
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=output/data.yaml, degrees=20.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, 